# Exploring 10K Generated Models: Gemma 4 vs Llama 3.2

This notebook loads pre-generated NumPyro model codes from two LLMs (Gemma 4 E4B and Llama 3.2),
runs Bayesian inference at multiple model pool sizes, and compares posteriors, model weights,
and epistemic uncertainty.

In [1]:
import json
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from pathlib import Path

import llb

plt.rcParams.update({
  "figure.figsize": (12, 5),
  "axes.titlesize": 13,
  "axes.labelsize": 11,
  "font.size": 10,
})

text = "I have a bunch of coin flips. What's the bias?"
data = {"flips": [0, 1, 0, 1, 1, 0]}
targets = ["true_bias"]

CACHES = {
  "Gemma 4 E4B": "model_cache/gemma4_e4b",
  "Llama 3.2": "model_cache/llama32",
}

all_codes = {}
for name, cache_dir in CACHES.items():
  merged_files = sorted(glob.glob(f"{cache_dir}/merged_*.json"))
  with open(merged_files[0]) as f:
    d = json.load(f)
  all_codes[name] = d["model_codes"]
  print(f"{name}: {len(all_codes[name])} model codes loaded")

/home/edmondcunnin_umass_edu/Large-Language-Bayes-Model/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Gemma 4 E4B: 10000 model codes loaded
Llama 3.2: 10000 model codes loaded


In [6]:
def get_unique_codes(codes):
  """Deduplicate model codes, ignoring comments and whitespace."""
  seen = set()
  unique = []
  for c in codes:
    key = "\n".join(
      l.strip() for l in c.splitlines()
      if l.strip() and not l.strip().startswith("#")
    )
    if key not in seen:
      seen.add(key)
      unique.append(c)
  return unique

def code_to_model(code_str):
  """Turn a model code string into a callable model function."""
  namespace = {}
  exec(code_str, namespace)
  return namespace["model"]

unique_gemma = get_unique_codes(all_codes["Gemma 4 E4B"])
unique_llama = get_unique_codes(all_codes["Llama 3.2"])
print(f"Gemma unique models: {len(unique_gemma)}")
print(f"Llama unique models: {len(unique_llama)}")

print("\n" + "=" * 60)
print("GEMMA MODEL 0")
print("=" * 60)
print(unique_gemma[0])

print("\n" + "=" * 60)
print("GEMMA MODEL 1")
print("=" * 60)
print(unique_gemma[1])

# Make them callable and run MCMC standalone
model_a = code_to_model(unique_gemma[0])
model_b = code_to_model(unique_gemma[1])

import jax
import numpyro
from numpyro.infer import MCMC, NUTS

data = {"flips": jax.numpy.array([0, 1, 0, 1, 1, 0])}

kernel = NUTS(model_b)
mcmc = MCMC(kernel, num_warmup=200, num_samples=500)
mcmc.run(jax.random.PRNGKey(0), data)
mcmc.print_summary()

Gemma unique models: 8
Llama unique models: 1

GEMMA MODEL 0
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist

def model(data):
    flips = jnp.array(data["flips"])
    # Prior on the true bias (probability of heads)
    true_bias = numpyro.sample("true_bias", dist.Beta(1.0, 1.0))
    
    # Likelihood: Each flip is a Bernoulli trial governed by the true_bias
    with numpyro.plate("flips_sequence", flips.shape[0]):
        numpyro.sample(f"obs_flip_{i+1}", dist.Bernoulli(probs=true_bias), obs=flips[i])
        for i in range(flips.shape[0]):
            pass # Plate context handles the sampling for all flips

GEMMA MODEL 1
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist

def model(data):
    # Model the coin flip bias (p)
    true_bias = numpyro.sample("true_bias", dist.Beta(1.0, 1.0))
    
    # Observed flips are Bernoulli trials based on the true_bias
    obs_flips = jnp.array(data["flips"])
    with numpyro.plate("flips", data

sample: 100%|██████████| 700/700 [00:00<00:00, 837.42it/s, 1 steps of size 9.97e-01. acc. prob=0.93] 


                 mean       std    median      5.0%     95.0%     n_eff     r_hat
  true_bias      0.49      0.17      0.49      0.21      0.74    194.61      1.00

Number of divergences: 0


## Run Inference at Multiple Scales

Sample subsets of increasing size from the 10K pool and run full BMA + LOO inference on each.

In [2]:
N_MODELS_LIST = [10, 50, 200, 1000]
SEED = 42

results = {}

for llm_name, codes in all_codes.items():
  rng = np.random.default_rng(SEED)
  for n in N_MODELS_LIST:
    print(f"\n{'='*60}")
    print(f"{llm_name}, n_models={n}")
    print(f"{'='*60}")
    idx = rng.choice(len(codes), size=n, replace=False)
    sampled = [codes[i] for i in idx]
    result = llb.infer(
      text, data, targets,
      model_codes=sampled,
      n_models=n,
      mcmc_num_warmup=200,
      mcmc_num_samples=500,
      loo_num_warmup=50,
      loo_num_samples=100,
      use_true_loo=True,
      verbose=False,
      auto_print_result=False,
    )
    results[(llm_name, n)] = result
    diag = result["diagnostics"]
    print(f"  Valid models: {diag['valid_models_final']}/{n}")
    print(f"  Epistemic var (LOO): {float(result['epistemic_uncertainty_loo']['true_bias']):.6f}")

print("\nAll runs complete.")

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.



Gemma 4 E4B, n_models=10
Computing TRUE LOO-ELBO for 6 datapoints (this will take a while)...
  LOO for datapoint 1/6... ELBO = -9.3859 (±3.8567)
  LOO for datapoint 2/6... ELBO = -9.5436 (±1.9382)
  LOO for datapoint 3/6... ELBO = -9.8046 (±2.1549)
  LOO for datapoint 4/6... ELBO = -10.7734 (±4.7789)
  LOO for datapoint 5/6... ELBO = -10.4537 (±2.7237)
  LOO for datapoint 6/6... ELBO = -10.1637 (±2.4621)


────────────────────────────────────────────── LOO Matrix Diagnostic ──────────────────────────────────────────────

Shape: (6, 1)


First 3 rows (first 3 datapoints):
[[-9.38594443]
 [-9.54356657]
 [-9.80455604]]

Variance per datapoint (across models):


...

L1 distance: 0.000000

Final LOO objective: -9.5712

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

  Valid models: 1/50
  Epistemic var (LOO): 0.000000

Llama 3.2, n_models=200
Computing TRUE LOO-ELBO for 6 datapoints (this will take a while)...
  LOO for datapoint 1/6... ELBO = -10.1342 (±2.9808)
  LOO for datapoint 2/6... ELBO = -8.9439 (±2.0898)
  LOO for datapoint 3/6... ELBO = -10.5669 (±2.2966)
  LOO for datapoint 4/6... ELBO = -9.2213 (±1.7678)
  LOO for datapoint 5/6... ELBO = -8.8213 (±2.1174)
  LOO for datapoint 6/6... 

KeyboardInterrupt: 

## Diagnostics Summary

Compile failures, inference failures, deduplication, and valid model counts for every run.

In [ ]:
rows = []
for (llm_name, n), res in results.items():
  d = res["diagnostics"]
  rows.append({
    "LLM": llm_name,
    "Requested": d["requested_models"],
    "Generated": d["generated_models"],
    "Deduplicated": d["deduplicated_models"],
    "Compile Fail": d["compile_failures"],
    "Inference Fail": d["inference_failures"],
    "Shape Mismatch": d["shape_mismatch_drops"],
    "Valid": d["valid_models_final"],
    "Valid Rate": f"{d['valid_models_final'] / d['requested_models']:.1%}",
  })

df_diag = pd.DataFrame(rows)
df_diag

## Posterior Distributions

For the largest model count, overlay the flat, BMA, and LOO posterior densities of `true_bias`.

In [ ]:
n_max = max(N_MODELS_LIST)
llm_names = list(all_codes.keys())

fig, axes = plt.subplots(1, len(llm_names), figsize=(14, 5), sharey=True)

for ax, llm_name in zip(axes, llm_names):
  res = results[(llm_name, n_max)]
  x_grid = np.linspace(0, 1, 300)

  for label, key in [("Flat (uniform)", "posterior_flat"),
                     ("BMA", "posterior_bma"),
                     ("LOO Stacking", "posterior_loo")]:
    samples = np.array(res[key]["true_bias"])
    samples = samples[(samples > 0) & (samples < 1)]
    if len(samples) > 10:
      kde = gaussian_kde(samples)
      ax.plot(x_grid, kde(x_grid), label=label)

  ax.axvline(np.mean(data["flips"]), color="black", linestyle="--", alpha=0.5, label="Observed rate")
  ax.set_title(f"{llm_name} (n={n_max})")
  ax.set_xlabel("true_bias")
  ax.legend(fontsize=9)

axes[0].set_ylabel("Density")
fig.suptitle("Posterior Distributions of Coin Bias", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## Epistemic Uncertainty vs Model Count

How does between-model variance change as we add more models to the pool?

In [ ]:
fig, axes = plt.subplots(1, len(llm_names), figsize=(14, 5), sharey=True)

for ax, llm_name in zip(axes, llm_names):
  ns = []
  epi_uniform, epi_bma, epi_loo = [], [], []
  for n in N_MODELS_LIST:
    res = results[(llm_name, n)]
    ns.append(n)
    epi_uniform.append(float(res["epistemic_uncertainty_uniform"]["true_bias"]))
    epi_bma.append(float(res["epistemic_uncertainty_bma"]["true_bias"]))
    epi_loo.append(float(res["epistemic_uncertainty_loo"]["true_bias"]))

  ax.plot(ns, epi_uniform, "o-", label="Uniform")
  ax.plot(ns, epi_bma, "s-", label="BMA")
  ax.plot(ns, epi_loo, "^-", label="LOO Stacking")
  ax.set_xscale("log")
  ax.set_xlabel("Number of models")
  ax.set_title(llm_name)
  ax.legend(fontsize=9)

axes[0].set_ylabel("Epistemic variance (true_bias)")
fig.suptitle("Epistemic Uncertainty vs Model Pool Size", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## Weight Distributions

For the largest model count, how concentrated are the BMA and LOO weights?

In [ ]:
fig, axes = plt.subplots(2, len(llm_names), figsize=(14, 8), sharey="row")

for col, llm_name in enumerate(llm_names):
  res = results[(llm_name, n_max)]
  w_bma = res["weights_bma"]
  w_loo = res["weights_loo"]

  axes[0, col].hist(w_bma, bins=50, edgecolor="white", alpha=0.8)
  axes[0, col].axvline(1.0 / len(w_bma), color="red", linestyle="--", alpha=0.6, label="Uniform")
  axes[0, col].set_title(f"{llm_name} BMA weights")
  axes[0, col].legend(fontsize=9)

  axes[1, col].hist(w_loo, bins=50, edgecolor="white", alpha=0.8)
  axes[1, col].axvline(1.0 / len(w_loo), color="red", linestyle="--", alpha=0.6, label="Uniform")
  axes[1, col].set_title(f"{llm_name} LOO weights")
  axes[1, col].legend(fontsize=9)

axes[0, 0].set_ylabel("Count (BMA)")
axes[1, 0].set_ylabel("Count (LOO)")
for ax in axes[1, :]:
  ax.set_xlabel("Weight")

fig.suptitle(f"Model Weight Distributions (n={n_max})", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## Gemma 4 vs Llama 3.2 Direct Comparison

Side-by-side summary at a fixed model count.

In [ ]:
n_compare = 200

rows = []
for llm_name in llm_names:
  res = results[(llm_name, n_compare)]
  d = res["diagnostics"]
  w_bma = res["weights_bma"]
  w_loo = res["weights_loo"]
  w_uni = res["weights_uniform"]

  rows.append({
    "LLM": llm_name,
    "Valid Models": d["valid_models_final"],
    "Posterior Mean (flat)": f"{np.mean(res['posterior_flat']['true_bias']):.4f}",
    "Posterior Mean (BMA)": f"{np.mean(res['posterior_bma']['true_bias']):.4f}",
    "Posterior Mean (LOO)": f"{np.mean(res['posterior_loo']['true_bias']):.4f}",
    "Epi Var (uniform)": f"{float(res['epistemic_uncertainty_uniform']['true_bias']):.6f}",
    "Epi Var (BMA)": f"{float(res['epistemic_uncertainty_bma']['true_bias']):.6f}",
    "Epi Var (LOO)": f"{float(res['epistemic_uncertainty_loo']['true_bias']):.6f}",
    "ESS (BMA)": f"{1 / np.sum(w_bma**2):.1f}",
    "ESS (LOO)": f"{1 / np.sum(w_loo**2):.1f}",
    "Entropy (BMA)": f"{-np.sum(w_bma * np.log(w_bma + 1e-10)):.3f}",
    "Entropy (LOO)": f"{-np.sum(w_loo * np.log(w_loo + 1e-10)):.3f}",
    "L1(LOO, BMA)": f"{np.sum(np.abs(w_loo - w_bma)):.4f}",
  })

df_compare = pd.DataFrame(rows).set_index("LLM").T
df_compare